In [1]:
from pyspark.sql import functions as F

# ============================================================
# AirOps 360 - W5-04
# Gold MVP: flight-grain fact + daily origin-airport aggregate
# ============================================================

SILVER_SOURCE = "lh_airops_silver.slv_flights_weather_enriched"

FACT_TABLE = "fact_flight_performance"
AGG_TABLE = "agg_daily_origin_airport_performance"

EXPECTED_FLIGHT_ROWS = 597_919
EXPECTED_WEATHER_MATCHES = 59_813

src = spark.table(SILVER_SOURCE)

required_cols = [
    "flight_key",
    "flight_date",
    "reporting_airline",
    "flight_number",
    "origin",
    "dest",

    "crs_dep_time_hhmm",
    "dep_time_hhmm",
    "crs_arr_time_hhmm",
    "arr_time_hhmm",

    "dep_delay_minutes_signed",
    "arr_delay_minutes_signed",
    "dep_del15",
    "arr_del15",

    "cancelled",
    "diverted",
    "air_time_minutes",
    "distance_miles",

    "carrier_delay_minutes",
    "weather_delay_minutes",
    "nas_delay_minutes",
    "security_delay_minutes",
    "late_aircraft_delay_minutes",

    "origin_weather_key",
    "origin_weather_hour_local",
    "origin_temperature_2m_c",
    "origin_precipitation_mm",
    "origin_snowfall_cm",
    "origin_weather_code",
    "origin_wind_speed_10m_kmh",
    "origin_weather_match_status",

    "_bronze_run_id",
]

missing_cols = sorted(set(required_cols) - set(src.columns))

assert not missing_cols, (
    f"STOP: required Silver columns missing: {missing_cols}"
)

source_rows = src.count()
source_distinct_keys = src.select("flight_key").distinct().count()

assert source_rows == EXPECTED_FLIGHT_ROWS
assert source_distinct_keys == EXPECTED_FLIGHT_ROWS

print("SOURCE READY")
print(f"Rows:                 {source_rows:,}")
print(f"Distinct flight_key:  {source_distinct_keys:,}")
print(f"Missing columns:      {missing_cols}")

StatementMeta(, fbb96a8f-4468-4668-9a51-df07603db138, 3, Finished, Available, Finished, False)

SOURCE READY
Rows:                 597,919
Distinct flight_key:  597,919
Missing columns:      []


In [2]:
# ============================================================
# BUILD FACT_FLIGHT_PERFORMANCE
#
# Grain:
#   1 row = 1 accepted scheduled flight occurrence
#   identified by flight_key
# ============================================================

def hhmm4(col_name):
    """
    Convert integer HHMM to a 4-character string.
    Examples:
        855  -> 0855
        1645 -> 1645
        NULL -> NULL
    """
    return (
        F.when(
            F.col(col_name).isNull(),
            F.lit(None).cast("string")
        )
        .otherwise(
            F.lpad(
                F.col(col_name).cast("string"),
                4,
                "0"
            )
        )
    )


origin_code = F.upper(F.trim(F.col("origin")))
dest_code = F.upper(F.trim(F.col("dest")))
carrier_code = F.upper(F.trim(F.col("reporting_airline")))


fact = (
    src
    .select(
        # Deterministic flight identity
        F.col("flight_key"),

        # Date
        F.date_format(
            F.col("flight_date"),
            "yyyyMMdd"
        ).cast("int").alias("date_key"),

        F.col("flight_date"),

        # Stable MVP surrogate keys.
        # The dimensions built later must use the same key logic.
        F.xxhash64(origin_code).alias("origin_airport_key"),
        F.xxhash64(dest_code).alias("destination_airport_key"),
        F.xxhash64(carrier_code).alias("carrier_key"),

        # Keep business keys for traceability / current MVP aggregate
        origin_code.alias("origin_airport_code"),
        dest_code.alias("destination_airport_code"),
        carrier_code.alias("carrier_code"),

        F.col("flight_number")
         .cast("string")
         .alias("flight_number"),

        # Scheduled / actual times
        hhmm4("crs_dep_time_hhmm").alias("crs_departure_time"),
        hhmm4("dep_time_hhmm").alias("actual_departure_time"),
        hhmm4("crs_arr_time_hhmm").alias("crs_arrival_time"),
        hhmm4("arr_time_hhmm").alias("actual_arrival_time"),

        # Signed delay preserves early flights as negative values
        F.col("dep_delay_minutes_signed")
         .alias("departure_delay_minutes"),

        F.col("arr_delay_minutes_signed")
         .alias("arrival_delay_minutes"),

        # Keep BTS 15-minute flags for business-rate denominators
        F.col("dep_del15")
         .alias("departure_delayed_15_flag"),

        F.col("arr_del15")
         .alias("arrival_delayed_15_flag"),

        # Operational state
        F.col("cancelled").alias("cancelled_flag"),
        F.col("diverted").alias("diverted_flag"),

        F.col("air_time_minutes"),
        F.col("distance_miles").alias("distance"),

        # BTS classified delay minutes
        F.col("carrier_delay_minutes"),
        F.col("weather_delay_minutes"),
        F.col("nas_delay_minutes"),
        F.col("security_delay_minutes"),
        F.col("late_aircraft_delay_minutes"),

        # Validated origin weather enrichment.
        # Keep explicit units from Silver.
        F.col("origin_weather_hour_local"),
        F.col("origin_temperature_2m_c"),
        F.col("origin_precipitation_mm"),
        F.col("origin_snowfall_cm"),
        F.col("origin_weather_code"),
        F.col("origin_wind_speed_10m_kmh"),

        F.col("origin_weather_match_status"),

        F.col("origin_weather_key")
         .isNotNull()
         .alias("weather_match_flag"),

        # Flight-side lineage
        F.col("_bronze_run_id").alias("load_run_id"),

        F.lit("gold_mvp_v0.1").alias("_gold_version"),
        F.current_timestamp().alias("_gold_published_at_utc"),
    )
)

print("Fact dataframe created.")
print(f"Columns: {len(fact.columns)}")

StatementMeta(, fbb96a8f-4468-4668-9a51-df07603db138, 4, Finished, Available, Finished, False)

Fact dataframe created.
Columns: 38


In [3]:
# ============================================================
# WRITE FACT + CRITICAL GOLD GRAIN CHECKS
# ============================================================

(
    fact.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FACT_TABLE)
)

fact_gold = spark.table(FACT_TABLE)

fact_rows = fact_gold.count()

fact_distinct_keys = (
    fact_gold
    .select("flight_key")
    .distinct()
    .count()
)

duplicate_flight_groups = (
    fact_gold
    .groupBy("flight_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_dimension_keys = (
    fact_gold
    .filter(
        F.col("date_key").isNull()
        | F.col("origin_airport_key").isNull()
        | F.col("destination_airport_key").isNull()
        | F.col("carrier_key").isNull()
    )
    .count()
)

weather_matches = (
    fact_gold
    .filter(F.col("weather_match_flag") == True)
    .count()
)


# ------------------------------------------------------------
# Observed surrogate-key collision checks
# ------------------------------------------------------------

origin_codes = (
    fact_gold
    .select("origin_airport_code")
    .distinct()
    .count()
)

origin_keys = (
    fact_gold
    .select("origin_airport_key")
    .distinct()
    .count()
)

dest_codes = (
    fact_gold
    .select("destination_airport_code")
    .distinct()
    .count()
)

dest_keys = (
    fact_gold
    .select("destination_airport_key")
    .distinct()
    .count()
)

carrier_codes = (
    fact_gold
    .select("carrier_code")
    .distinct()
    .count()
)

carrier_keys = (
    fact_gold
    .select("carrier_key")
    .distinct()
    .count()
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert fact_rows == EXPECTED_FLIGHT_ROWS
assert fact_distinct_keys == EXPECTED_FLIGHT_ROWS
assert duplicate_flight_groups == 0
assert null_dimension_keys == 0

assert weather_matches == EXPECTED_WEATHER_MATCHES

assert origin_codes == origin_keys
assert dest_codes == dest_keys
assert carrier_codes == carrier_keys


print("=" * 70)
print("AIR0PS 360 - W5-04 GOLD FACT EVIDENCE")
print("=" * 70)

print(f"Gold fact rows:             {fact_rows:,}")
print(f"Distinct flight_key:        {fact_distinct_keys:,}")
print(f"Duplicate flight groups:    {duplicate_flight_groups:,}")
print(f"NULL dimension keys:        {null_dimension_keys:,}")

print()
print(f"Weather-matched flights:    {weather_matches:,}")

print()
print(f"Origin codes / keys:        {origin_codes:,} / {origin_keys:,}")
print(f"Destination codes / keys:   {dest_codes:,} / {dest_keys:,}")
print(f"Carrier codes / keys:       {carrier_codes:,} / {carrier_keys:,}")

print()
print("FACT GRAIN STATUS: PASS")
print("=" * 70)

StatementMeta(, fbb96a8f-4468-4668-9a51-df07603db138, 5, Finished, Available, Finished, False)

AIR0PS 360 - W5-04 GOLD FACT EVIDENCE
Gold fact rows:             597,919
Distinct flight_key:        597,919
Duplicate flight groups:    0
NULL dimension keys:        0

Weather-matched flights:    59,813

Origin codes / keys:        342 / 342
Destination codes / keys:   342 / 342
Carrier codes / keys:       13 / 13

FACT GRAIN STATUS: PASS


In [4]:
# ============================================================
# DAILY ORIGIN-AIRPORT PERFORMANCE AGGREGATE
#
# Grain:
#   1 row = 1 flight_date + 1 origin airport
# ============================================================

daily_airport = (
    fact_gold

    .groupBy(
        "date_key",
        "flight_date",
        "origin_airport_key",
        "origin_airport_code",
    )

    .agg(
        # Volume
        F.count("*").alias("total_flights"),

        # Cancellation / diversion
        F.sum(
            F.col("cancelled_flag").cast("long")
        ).alias("cancelled_flights"),

        F.sum(
            F.col("diverted_flag").cast("long")
        ).alias("diverted_flights"),

        # IMPORTANT:
        # arr_del15 can be NULL, especially when a normal
        # arrival-delay outcome does not exist.
        F.sum(
            F.when(
                F.col("arrival_delayed_15_flag").isNotNull(),
                1
            ).otherwise(0)
        ).alias("arrival_delay_eligible_flights"),

        F.sum(
            F.when(
                F.col("arrival_delayed_15_flag") == True,
                1
            ).otherwise(0)
        ).alias("arrival_delayed_15_flights"),

        # Signed delay: early arrivals may reduce the average
        F.avg(
            "arrival_delay_minutes"
        ).alias("avg_arrival_delay_minutes"),
    )

    # Cancellation denominator = all scheduled flights
    .withColumn(
        "cancellation_rate",
        F.round(
            F.col("cancelled_flights")
            / F.col("total_flights"),
            4
        )
    )

    # Arrival-delay denominator = flights for which arr_del15
    # is actually known.
    .withColumn(
        "arrival_delay_15_rate",
        F.when(
            F.col("arrival_delay_eligible_flights") > 0,

            F.round(
                F.col("arrival_delayed_15_flights")
                / F.col("arrival_delay_eligible_flights"),
                4
            )
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )

    .withColumn(
        "_gold_version",
        F.lit("gold_mvp_v0.1")
    )

    .withColumn(
        "_gold_published_at_utc",
        F.current_timestamp()
    )
)


(
    daily_airport.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(AGG_TABLE)
)


agg_gold = spark.table(AGG_TABLE)


# ============================================================
# AGGREGATE GRAIN + RECONCILIATION
# ============================================================

aggregate_rows = agg_gold.count()

duplicate_aggregate_groups = (
    agg_gold
    .groupBy(
        "flight_date",
        "origin_airport_code"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

reconciled_flights = (
    agg_gold
    .agg(
        F.sum("total_flights").alias("n")
    )
    .first()["n"]
)

reconciled_cancellations = (
    agg_gold
    .agg(
        F.sum("cancelled_flights").alias("n")
    )
    .first()["n"]
)

fact_cancellations = (
    fact_gold
    .filter(
        F.col("cancelled_flag") == True
    )
    .count()
)


assert duplicate_aggregate_groups == 0
assert reconciled_flights == fact_rows
assert reconciled_cancellations == fact_cancellations


# ============================================================
# ONE EXPECTED AGGREGATE CHECK
#
# Pick the largest airport-day group, independently recalculate
# it from the fact, and compare it to the stored aggregate.
# ============================================================

sample = (
    agg_gold
    .orderBy(
        F.desc("total_flights"),
        "flight_date",
        "origin_airport_code"
    )
    .first()
)

expected = (
    fact_gold
    .filter(
        (F.col("flight_date") == sample["flight_date"])
        &
        (
            F.col("origin_airport_code")
            == sample["origin_airport_code"]
        )
    )
    .agg(
        F.count("*").alias("expected_total_flights"),

        F.sum(
            F.col("cancelled_flag").cast("long")
        ).alias("expected_cancelled_flights")
    )
    .first()
)


assert (
    sample["total_flights"]
    == expected["expected_total_flights"]
)

assert (
    sample["cancelled_flights"]
    == expected["expected_cancelled_flights"]
)


print("=" * 70)
print("AIR0PS 360 - W5-04 DAILY AIRPORT AGGREGATE EVIDENCE")
print("=" * 70)

print(f"Aggregate rows:                 {aggregate_rows:,}")
print(f"Duplicate airport-day groups:   {duplicate_aggregate_groups:,}")
print(f"Sum(total_flights):             {reconciled_flights:,}")
print(f"Fact rows:                      {fact_rows:,}")

print()
print("EXPECTED AGGREGATE CHECK")
print("------------------------")
print(f"Date:                            {sample['flight_date']}")
print(f"Origin:                          {sample['origin_airport_code']}")
print(f"Stored total flights:            {sample['total_flights']:,}")
print(f"Expected total flights:          {expected['expected_total_flights']:,}")
print(f"Stored cancellations:            {sample['cancelled_flights']:,}")
print(f"Expected cancellations:          {expected['expected_cancelled_flights']:,}")

print()
print("AGGREGATE STATUS: PASS")
print("=" * 70)


display(
    agg_gold
    .orderBy(
        F.desc("total_flights")
    )
    .limit(20)
)

StatementMeta(, fbb96a8f-4468-4668-9a51-df07603db138, 6, Finished, Available, Finished, False)

AIR0PS 360 - W5-04 DAILY AIRPORT AGGREGATE EVIDENCE
Aggregate rows:                 10,031
Duplicate airport-day groups:   0
Sum(total_flights):             597,919
Fact rows:                      597,919

EXPECTED AGGREGATE CHECK
------------------------
Date:                            2026-04-16
Origin:                          ORD
Stored total flights:            1,161
Expected total flights:          1,161
Stored cancellations:            26
Expected cancellations:          26

AGGREGATE STATUS: PASS


SynapseWidget(Synapse.DataFrame, 25a200ea-7838-49e8-9ee1-e0d3c8563102)